# Bitcoin Market Data Analysis

## Objective

This notebook investigates Bitcoin OHLC price data from the CoinGecko API. It demonstrates data collection, validation, transformation, exploratory analysis and communication of findings. The analysis is educational and is not financial advice.

## 1. Setup and data collection

The notebook reuses the application's API client and data-processing pipeline so the analysis is based on the same validated data displayed by the visualiser.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Allow the notebook to import modules from the repository root.
repository_root = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from api_client import fetch_data
from indicators import calculate_market_indicators

df = fetch_data(365)
if df.empty:
    raise RuntimeError("CoinGecko returned no data. Check the connection and try again.")

df.head()

## 2. Data validation

Before calculating metrics, the analysis checks structure, types, missing values, duplicates, timestamp order and invalid prices.

In [ ]:
quality_report = pd.Series({
    "rows": len(df),
    "columns": len(df.columns),
    "missing_values": int(df.isna().sum().sum()),
    "duplicate_timestamps": int(df.index.duplicated().sum()),
    "timestamps_in_order": bool(df.index.is_monotonic_increasing),
    "non_positive_prices": int((df[["Open", "High", "Low", "Close"]] <= 0).sum().sum()),
})

print(df.dtypes)
quality_report.to_frame("result")

In [ ]:
analysis_df = df.copy()
analysis_df = analysis_df[~analysis_df.index.duplicated(keep="last")].sort_index()
analysis_df = analysis_df.dropna(subset=["Open", "High", "Low", "Close"])
analysis_df = analysis_df[(analysis_df[["Open", "High", "Low", "Close"]] > 0).all(axis=1)]

observation_interval = analysis_df.index.to_series().diff().median()
print(f"Analysis period: {analysis_df.index.min()} to {analysis_df.index.max()}")
print(f"Rows after validation: {len(analysis_df)}")
print(f"Typical interval between observations: {observation_interval}")

## 3. Feature engineering

Returns are calculated between consecutive API observations. They are described as *period returns* because CoinGecko may change the interval between OHLC observations depending on the requested timeframe.

In [ ]:
analysis_df["Period Return (%)"] = analysis_df["Close"].pct_change() * 100
analysis_df["MA 7"] = analysis_df["Close"].rolling(window=7).mean()
analysis_df["MA 30"] = analysis_df["Close"].rolling(window=30).mean()
analysis_df["Rolling Volatility 30"] = analysis_df["Period Return (%)"].rolling(window=30).std()

indicators = calculate_market_indicators(analysis_df)
summary = pd.Series({
    "Latest close (USD)": indicators["latest_close"],
    "Highest price (USD)": indicators["highest_price"],
    "Lowest price (USD)": indicators["lowest_price"],
    "Return volatility (%)": indicators["volatility"],
    "Maximum drawdown (%)": indicators["max_drawdown"],
})
summary.to_frame("value").round(2)

## 4. Exploratory visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(analysis_df.index, analysis_df["Close"], label="Closing price", color="navy")
ax.set(title="Bitcoin closing price", xlabel="Date", ylabel="Price (USD)")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(analysis_df.index, analysis_df["Close"], label="Closing price", alpha=0.45)
ax.plot(analysis_df.index, analysis_df["MA 7"], label="7-observation moving average")
ax.plot(analysis_df.index, analysis_df["MA 30"], label="30-observation moving average")
ax.set(title="Bitcoin price and moving averages", xlabel="Date", ylabel="Price (USD)")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(analysis_df.index, analysis_df["Rolling Volatility 30"], color="darkred")
axes[0].set(title="30-observation rolling volatility", xlabel="Date", ylabel="Standard deviation (%)")
axes[0].grid(alpha=0.25)

axes[1].hist(analysis_df["Period Return (%)"].dropna(), bins=30, color="slateblue", edgecolor="white")
axes[1].set(title="Distribution of period returns", xlabel="Return (%)", ylabel="Frequency")

plt.tight_layout()
plt.show()

## 5. Evidence-based findings

Run the next cell to generate statements from the current API response rather than relying on hard-coded market claims.

In [ ]:
valid_returns = analysis_df["Period Return (%)"].dropna()
largest_gain_time = valid_returns.idxmax()
largest_loss_time = valid_returns.idxmin()
peak_volatility_time = analysis_df["Rolling Volatility 30"].idxmax()

print(f"1. The largest positive period return was {valid_returns.max():.2f}% at {largest_gain_time}.")
print(f"2. The largest negative period return was {valid_returns.min():.2f}% at {largest_loss_time}.")
print(f"3. Rolling volatility peaked at {analysis_df.loc[peak_volatility_time, 'Rolling Volatility 30']:.2f}% at {peak_volatility_time}.")
print(f"4. Maximum drawdown across the selected data was {indicators['max_drawdown']:.2f}%.")

## 6. Interpretation and limitations

The shorter moving average reacts more quickly to changes, while the longer moving average smooths short-term variation. Rolling volatility highlights periods when returns became less stable, and maximum drawdown measures the largest decline from an earlier peak.

Limitations:

- Historical prices do not predict future prices.
- Results depend on the timeframe and observation frequency returned by CoinGecko.
- The OHLC endpoint used by this application does not provide genuine volume data.
- Price data alone cannot explain the economic, regulatory or behavioural causes of market changes.
- This is an educational analysis, not financial advice.


## 7. Possible extensions

Future work could compare several assets, store observations in a database, add reproducible data snapshots or evaluate a simple forecasting baseline with a chronological train/test split. Any predictive work should be compared against a naive baseline and assessed for data leakage.